# Aula Prática — Introdução ao Apache Kafka
### Computação Paralela e Distribuída (DCE540)

Nesta aula vamos:

1. Entender os conceitos fundamentais do Kafka;
2. Subir um cluster Kafka local via Docker;
3. Criar tópicos e produzir/consumir mensagens em Python;
4. Realizar duas atividades práticas que ilustram paralelismo e distribuição via *consumer groups* e *partições*.

> **Pré-requisitos:** VS Code (com as extensões Python e Jupyter) e Docker/Docker Compose instalados na máquina onde este notebook será executado.


## 0. Preparando o ambiente (VS Code + Docker)

Este notebook foi pensado para rodar no **VS Code**, usando as extensões **Python** e **Jupyter**. Siga estes passos antes de começar:

1. Instale o **VS Code** (se ainda não estiver instalado no laboratório).
2. Na aba de Extensões do VS Code, instale:
   - **Python** (da Microsoft);
   - **Jupyter** (da Microsoft).
3. Abra um terminal (menu **Terminal > New Terminal** no VS Code) e crie um ambiente virtual Python na pasta desta aula:

   **Windows (PowerShell):**
   ```powershell
   python -m venv venv
   venv\Scripts\activate
   ```

   **Linux/Mac:**
   ```bash
   python3 -m venv venv
   source venv/bin/activate
   ```

   > No Windows, o comando costuma ser `python` (não `python3`). Se `python -m venv venv` também não funcionar, rode `python --version` para confirmar que o Python está instalado e acessível nesse terminal antes de continuar.

4. Instale as dependências Python desta aula:

   ```bash
   pip install --upgrade pip
   pip install -r requirements.txt
   ```

5. Abra o arquivo `aula_kafka_dce540.ipynb` no VS Code (clique duas vezes nele no explorador de arquivos).
6. No canto superior direito do notebook, clique em **Select Kernel** e escolha o ambiente `venv` criado no passo 3.
7. Confirme que o **Docker** está instalado e rodando (ícone da baleia na barra de tarefas, ou `docker --version` no terminal).
8. Ainda no terminal do VS Code, na pasta desta aula, suba o ambiente Kafka:

   ```bash
   docker compose up -d
   ```

   Aguarde ~15-20 segundos para o broker inicializar completamente. Para conferir:

   ```bash
   docker compose ps
   docker compose logs kafka --tail=50
   ```

9. (Opcional) Acompanhe visualmente os tópicos/partições/mensagens em `http://localhost:8080` (Kafka UI, já incluso no `docker-compose.yml`).

A partir daqui, execute as células do notebook em ordem (`Shift+Enter` em cada uma).

Ao final da aula, encerre o ambiente Kafka:

```bash
docker compose down -v
```


## 1. Conceitos e definições

| Termo | Definição |
|---|---|
| **Broker** | Um servidor Kafka. Recebe, armazena e entrega mensagens. Um cluster Kafka é formado por um ou mais brokers. |
| **Cluster** | Conjunto de brokers trabalhando juntos, coordenando a distribuição e replicação dos dados. |
| **KRaft** | Modo de operação do Kafka em que o próprio cluster gerencia metadados via um protocolo de consenso (Raft), eliminando a dependência do ZooKeeper (usado em versões mais antigas). |
| **Topic (tópico)** | Um canal nomeado para onde mensagens são publicadas. Equivalente a uma "fila" lógica, mas pode ter múltiplos leitores independentes. |
| **Partition (partição)** | Cada tópico é dividido em uma ou mais partições. É a unidade de paralelismo do Kafka: partições diferentes podem ser processadas em paralelo por consumidores diferentes. |
| **Offset** | Identificador sequencial (inteiro) da posição de uma mensagem dentro de uma partição. Cada consumidor controla seu próprio offset de leitura. |
| **Producer (produtor)** | Aplicação/cliente que publica (escreve) mensagens em um tópico. |
| **Consumer (consumidor)** | Aplicação/cliente que lê mensagens de um tópico. |
| **Consumer Group (grupo de consumidores)** | Conjunto de consumidores que dividem entre si o trabalho de ler as partições de um tópico — cada partição é lida por apenas um consumidor do grupo por vez. É o principal mecanismo de paralelismo e balanceamento de carga do Kafka. |
| **Replication Factor (fator de replicação)** | Número de cópias de cada partição mantidas em brokers diferentes, para tolerância a falhas. |
| **Leader / Follower** | Para cada partição replicada, um broker é o *leader* (atende leituras/escritas) e os demais são *followers* (mantêm cópias sincronizadas). |

**Conexão com o que já vimos em MPI:** assim como processos MPI trocam mensagens diretamente (comunicação síncrona, acoplada), Kafka desacopla produtores e consumidores no tempo e no espaço — um produtor não precisa saber quem (ou quantos) vai consumir a mensagem, e a entrega é assíncrona e persistente.

### Por que dividir em tópicos e partições?

**Tópicos** existem para dar **organização lógica** ao fluxo de dados: cada tipo de evento ou fonte de dados tem seu próprio canal nomeado (ex.: `sensores-temperatura`, `pedidos-realizados`, `logs-erro`), permitindo que produtores e consumidores diferentes trabalhem de forma independente, sem interferir uns nos outros.

**Partições**, por sua vez, existem por dois motivos combinados:

1. **Paralelismo** — um tópico com 1 única partição só pode ser lido por 1 consumidor de cada vez dentro de um grupo, virando um gargalo. Ao dividir o tópico em várias partições, o Kafka distribui a carga entre múltiplos consumidores simultâneos (cada partição é atribuída a um consumidor diferente do grupo), exatamente como dividir um problema entre vários processos MPI para ganhar desempenho.
2. **Escalabilidade e distribuição física** — cada partição pode residir (e ser replicada) em um broker diferente do cluster. Isso significa que o volume de dados e a carga de leitura/escrita de um tópico não ficam limitados à capacidade de uma única máquina.

Em outras palavras: o **tópico** organiza *o quê* está sendo comunicado; a **partição** é o mecanismo que viabiliza processar isso em paralelo e de forma distribuída — os dois pilares centrais da disciplina.


## 2. Preparando o ambiente Python

Vamos usar a biblioteca `kafka-python`, um cliente Kafka escrito em Python puro (não depende de bibliotecas C externas), o que facilita a instalação em qualquer sistema operacional. Ela implementa o protocolo do Kafka e nos dá as classes que vamos usar ao longo da aula:

- `KafkaAdminClient` — para tarefas administrativas, como criar tópicos;
- `KafkaProducer` — para publicar mensagens;
- `KafkaConsumer` — para ler mensagens.

Se você já instalou as dependências pelo `requirements.txt` (Seção 0), a célula abaixo não faz nada de novo — é só uma garantia, útil caso alguém pule direto para esta célula.


In [ ]:
%pip install kafka-python --quiet


Com a biblioteca disponível, importamos as classes que vamos usar e definimos uma constante com o endereço do broker.

O valor `localhost:9094` não é arbitrário: é a porta do listener **EXTERNAL** que configuramos no `docker-compose.yml` (Seção 0), pensada justamente para clientes rodando **fora** dos containers — ou seja, este notebook, executando diretamente na sua máquina. A porta `9092` (listener **PLAINTEXT**) é usada para comunicação **entre** containers (por exemplo, o Kafka UI se conecta por ela) e não está acessível diretamente do host da mesma forma.


In [ ]:
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
from kafka.admin import NewTopic
import json
import time

BOOTSTRAP_SERVERS = "localhost:9094"  # listener EXTERNAL definido no docker-compose.yml


**Dica de depuração:** se a célula acima (ou qualquer célula seguinte que conecta ao Kafka) travar ou lançar um erro de conexão (`NoBrokersAvailable`), confira antes de continuar:

- o Docker Desktop está aberto e rodando (ícone da baleia estável na barra de tarefas)?
- os containers estão de pé? Rode `docker compose ps` no terminal — `kafka-aula` deve aparecer com status `running`;
- já se passaram uns 15-20 segundos desde o `docker compose up -d`? O broker leva um tempo para inicializar completamente.


## 3. Criando um tópico

Vamos criar um tópico `sensores-temperatura` com **3 partições** e fator de replicação **1** (só temos 1 broker neste ambiente de aula).

O número de partições é o que vai permitir que, mais adiante, múltiplos consumidores processem mensagens em paralelo.


In [ ]:
admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)

topic = NewTopic(name="sensores-temperatura", num_partitions=3, replication_factor=1)

try:
    admin.create_topics([topic])
    print("Tópico criado com sucesso.")
except Exception as e:
    print(f"Aviso: {e}")

print(admin.list_topics())
admin.close()


## 4. Producer — publicando mensagens

Vamos simular sensores de temperatura enviando leituras para o tópico. Repare que a **chave** (`key`) da mensagem é usada pelo Kafka para decidir em qual partição ela cai — mensagens com a mesma chave sempre vão para a mesma partição.


In [ ]:
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

import random

sensores = ["sensor-A", "sensor-B", "sensor-C"]

# Alterado para gerar >= 50 mensagens, conforme pedido na Atividade Prática 2 (Seção 7)
for i in range(60):
    sensor = random.choice(sensores)
    leitura = {
        "sensor": sensor,
        "temperatura_c": round(random.uniform(18, 35), 1),
        "timestamp": time.time(),
    }
    producer.send("sensores-temperatura", key=sensor, value=leitura)
    print(f"Enviado: {leitura}")
    time.sleep(0.2)

producer.flush()
producer.close()
print("Produção finalizada.")


## 5. Consumer — lendo mensagens

Um consumidor simples que lê todas as mensagens do início do tópico.


In [ ]:
consumer = KafkaConsumer(
    "sensores-temperatura",
    bootstrap_servers=BOOTSTRAP_SERVERS,
    auto_offset_reset="earliest",   # lê desde o início do tópico
    enable_auto_commit=True,
    group_id="grupo-leitura-simples",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=5000,       # encerra se ficar 5s sem novas mensagens
)

for msg in consumer:
    print(f"Partição {msg.partition} | offset {msg.offset} | {msg.value}")

consumer.close()


## 6. Atividade Prática 1 — Consumer Group e paralelismo

**Objetivo:** observar como o Kafka distribui as 3 partições do tópico `sensores-temperatura` entre múltiplos consumidores de um mesmo *consumer group*.

**Passo a passo:**

1. Reexecute a célula do Producer (seção 4) para gerar novas mensagens.
2. Abra **dois terminais** separados (fora deste notebook) e, em cada um, rode um script Python igual ao da seção 6, mas:
   - use o **mesmo** `group_id` (ex.: `"grupo-paralelo"`) nos dois;
   - remova o `consumer_timeout_ms` (ou aumente bastante), para o consumidor ficar escutando continuamente.
3. Rode o Producer novamente e observe: cada mensagem é entregue a **apenas um** dos dois consumidores.
4. Anote: quantas partições cada consumidor recebeu? O que acontece se você abrir um **terceiro** consumidor no mesmo grupo? E um **quarto** (mais consumidores que partições)?

**Pergunta para o relatório:** por que o Kafka não permite mais consumidores ativos, em um mesmo grupo, do que o número de partições do tópico? Relacione isso com os limites de paralelismo que já vimos em MPI (número de partições ≈ número de processos que podem trabalhar simultaneamente).


In [ ]:
# Célula auxiliar para a Atividade 1:
# Rode esta célula em DUAS instâncias separadas deste notebook (ou copie para dois scripts .py),
# com o mesmo group_id, e compare quais partições cada uma recebe.

GROUP_ID = "grupo-paralelo"  # use o MESMO valor nos dois consumidores

consumer = KafkaConsumer(
    "sensores-temperatura",
    bootstrap_servers=BOOTSTRAP_SERVERS,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id=GROUP_ID,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    # Timeout adicionado para não travar ao usar "Run All".
    # Para o uso original em terminais separados (ouvindo continuamente),
    # é só remover esta linha ou aumentar o valor.
    consumer_timeout_ms=15000,
)

print(f"Consumidor iniciado no grupo '{GROUP_ID}'. Aguardando mensagens...")
for msg in consumer:
    print(f"[{GROUP_ID}] Partição {msg.partition} | offset {msg.offset} | {msg.value}")


In [ ]:
import threading
import uuid

def rodar_consumidor(consumer_id, group_id, resultados, pronto_event, tempo_limite_ms=10000):
    """Entra no grupo, espera até receber uma atribuição de partições (fazendo poll()
    repetidamente), registra quais partições foram atribuídas e só então passa
    a contar as mensagens recebidas de fato."""
    consumer = KafkaConsumer(
        "sensores-temperatura",
        bootstrap_servers=BOOTSTRAP_SERVERS,
        auto_offset_reset="latest",  # só conta mensagens NOVAS, produzidas após o grupo estabilizar
        enable_auto_commit=True,
        group_id=group_id,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=tempo_limite_ms,
    )
    limite = time.time() + 15
    while not consumer.assignment() and time.time() < limite:
        consumer.poll(timeout_ms=500)
    particoes_atribuidas = sorted(p.partition for p in consumer.assignment())
    pronto_event.set()

    particoes_com_mensagem = set()
    total_mensagens = 0
    for msg in consumer:
        particoes_com_mensagem.add(msg.partition)
        total_mensagens += 1
    consumer.close()
    resultados[consumer_id] = {
        "particoes_atribuidas": particoes_atribuidas,
        "particoes_com_mensagem": sorted(particoes_com_mensagem),
        "total_mensagens": total_mensagens,
    }


def enviar_lote(n=30):
    """Producer auxiliar: envia um novo lote de leituras DEPOIS que os consumidores
    já estão no grupo, para que a distribuição aconteça ao vivo."""
    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        key_serializer=lambda k: k.encode("utf-8"),
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    )
    sensores = ["sensor-A", "sensor-B", "sensor-C"]
    for _ in range(n):
        sensor = random.choice(sensores)
        leitura = {
            "sensor": sensor,
            "temperatura_c": round(random.uniform(18, 35), 1),
            "timestamp": time.time(),
        }
        producer.send("sensores-temperatura", key=sensor, value=leitura)
        time.sleep(0.1)
    producer.flush()
    producer.close()


def testar_paralelismo(n_consumidores, group_id_base):
    # Sufixo único a cada chamada: evita herdar offsets committados de execuções
    # anteriores desta mesma célula (senão o grupo retoma de onde parou da última vez).
    group_id = f"{group_id_base}-{uuid.uuid4().hex[:6]}"

    resultados = {}
    eventos_prontos = []
    threads = []
    for i in range(n_consumidores):
        evento = threading.Event()
        eventos_prontos.append(evento)
        t = threading.Thread(
            target=rodar_consumidor,
            args=(f"consumidor-{i + 1}", group_id, resultados, evento),
        )
        threads.append(t)
        t.start()

    # Espera todos os consumidores receberem sua atribuição de partições
    for evento in eventos_prontos:
        evento.wait(timeout=15)
    time.sleep(2)  # margem extra para o rebalanceamento terminar de fato

    print(f"Consumidores prontos, enviando novo lote de mensagens para o grupo '{group_id}'...")
    enviar_lote(30)

    for t in threads:
        t.join()

    print(f"\n=== Resultado com {n_consumidores} consumidor(es) (grupo '{group_id}') ===")
    for cid, info in resultados.items():
        print(
            f"{cid}: partições atribuídas {info['particoes_atribuidas']} | "
            f"partições com mensagem {info['particoes_com_mensagem']} | "
            f"{info['total_mensagens']} mensagens"
        )
    return resultados


print(">>> Teste com 2 consumidores (menos consumidores que partições)")
testar_paralelismo(2, group_id_base="grupo-paralelo-2")

print("\n>>> Teste com 3 consumidores (consumidores = número de partições)")
testar_paralelismo(3, group_id_base="grupo-paralelo-3")

print("\n>>> Teste com 4 consumidores (mais consumidores que partições)")
testar_paralelismo(4, group_id_base="grupo-paralelo-4")


### Resposta — Atividade Prática 1

Rodando os testes, deu pra ver bem na prática como o Kafka distribui as partições entre os consumidores de um mesmo grupo.

Com **2 consumidores**, o Kafka dividiu as 3 partições entre os dois: um ficou com as partições 0 e 1 e recebeu 17 mensagens (mas só vindas da partição 0, porque a 1 não teve nenhuma leitura nesse lote), e o outro ficou só com a partição 2 e recebeu 13 mensagens.

Com **3 consumidores**, cada um ficou com exatamente uma partição, que é o cenário ideal: um pegou a partição 0 e recebeu 24 mensagens, outro pegou a 2 e recebeu 6, e o terceiro ficou com a partição 1 — que de novo não teve nenhuma mensagem naquele lote, então apareceu com 0. Isso não significa que ele ficou de fora do grupo, só que a partição dele estava "quieta" naquele momento (questão de sorte no hash das chaves, não erro).

Já com **4 consumidores**, ou seja, mais gente que partição, necessariamente alguém fica sem receber nada — isso é esperado. No meu teste isso ficou até mais evidente do que eu esperava: como o Kafka reorganiza (rebalanceia) o grupo toda vez que um novo consumidor entra, e isso foi acontecendo um atrás do outro dentro do mesmo processo, o último consumidor a entrar acabou herdando praticamente todas as partições ativas e ficou com as 30 mensagens sozinho, enquanto os outros três fecharam com 0. Isso é um efeito de timing de rodar tudo simulado em threads no mesmo notebook (em terminais separados de verdade a distribuição costuma ficar mais uniforme), mas o ponto principal continua valendo: nunca teve mais de 3 "fluxos" de mensagem rolando ao mesmo tempo, e sempre sobra consumidor sem receber nada quando tem mais consumidor do que partição.

**Por que o Kafka não deixa mais consumidores ativos do que partições?**

Porque dentro de um *consumer group*, cada partição só pode ser atribuída a um consumidor por vez. Isso existe pra manter a ordem das mensagens dentro da partição e evitar que dois consumidores processem (e façam commit) da mesma mensagem ao mesmo tempo, o que ia gerar processamento duplicado ou inconsistente. Como a partição é a menor unidade que o Kafka consegue distribuir, ela acaba virando o teto do paralelismo real do grupo: se eu colocar mais consumidores do que partições, o excedente simplesmente não tem pra onde ir e fica esperando à toa.

Isso lembra bastante o que vimos em MPI quando dividimos o trabalho em blocos e distribuímos um bloco por processo (como no exercício do Mandelbrot). Cada bloco só pode ir pra um processo de cada vez — não faz sentido dois processos mexendo no mesmo pedaço ao mesmo tempo. Se eu subo mais processos MPI do que blocos de trabalho, os processos que sobram ficam ociosos, exatamente como aconteceu com os consumidores extras aqui. No fim das contas, a unidade de divisão do trabalho (partição no Kafka, bloco no MPI) é o que realmente limita quanto paralelismo dá pra explorar — botar mais "trabalhador" além disso não ajuda em nada, só deixa gente parada.


## 7. Atividade Prática 2 — Agregação simples por sensor

**Objetivo:** implementar um consumidor que processa o fluxo de mensagens e calcula, para cada sensor, a temperatura média observada — um exemplo simples de processamento distribuído de fluxo (o que motiva ferramentas como Kafka Streams ou Spark Streaming, vistos mais adiante na disciplina).

**Tarefa:**

1. Complete o código abaixo para acumular as leituras por `sensor` e calcular a média a cada 5 mensagens recebidas.
2. Modifique o Producer da seção 4 para gerar pelo menos 50 mensagens.
3. Compare o resultado do seu consumidor com uma agregação feita "offline" (ex.: lendo tudo com Pandas) — os valores batem?

**Desafio opcional:** e se você rodar duas instâncias deste consumidor **no mesmo grupo**? A agregação de cada instância vai bater com o total? O que isso te diz sobre agregações *stateful* em sistemas distribuídos?


In [ ]:
from collections import defaultdict

consumer = KafkaConsumer(
    "sensores-temperatura",
    bootstrap_servers=BOOTSTRAP_SERVERS,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="grupo-agregacao",  # troque o group_id se quiser reler tudo do zero
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=8000,
)

soma = defaultdict(float)
contagem = defaultdict(int)

for msg in consumer:
    leitura = msg.value
    # Atualiza soma e contagem para o sensor da leitura atual
    soma[leitura["sensor"]] += leitura["temperatura_c"]
    contagem[leitura["sensor"]] += 1

    if sum(contagem.values()) % 5 == 0:
        medias = {s: round(soma[s] / contagem[s], 2) for s in soma if contagem[s] > 0}
        print(f"Médias parciais: {medias}")

consumer.close()
medias_finais = {s: round(soma[s] / contagem[s], 2) for s in soma if contagem[s] > 0}
print(f"Médias finais: {medias_finais}")


In [ ]:
%pip install pandas --quiet

import pandas as pd

# Agregação "offline": lê o tópico do início (novo group_id) e calcula as médias com Pandas,
# para comparar com o resultado do consumidor em streaming acima.
consumer_offline = KafkaConsumer(
    "sensores-temperatura",
    bootstrap_servers=BOOTSTRAP_SERVERS,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="grupo-comparacao-offline",  # grupo novo -> começa do zero, lê tudo
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=8000,
)

leituras = [msg.value for msg in consumer_offline]
consumer_offline.close()

df = pd.DataFrame(leituras)
medias_pandas = df.groupby("sensor")["temperatura_c"].mean().round(2).to_dict()

print(f"Total de leituras lidas: {len(df)}")
print(f"Médias por sensor (Pandas, offline): {medias_pandas}")
print(f"Médias por sensor (consumer em streaming, Seção 7): {medias_finais}")
print(f"Os valores batem? {medias_pandas == medias_finais}")


### Resposta — Atividade Prática 2

Rodando a agregação em streaming e depois comparando com o Pandas lendo tudo de novo do zero, os valores bateram certinho: sensor-A ficou em 27.0, sensor-B em 25.64 e sensor-C em 26.0, com 150 leituras no total. Faz sentido bater, porque as duas abordagens processam exatamente as mesmas mensagens — a diferença é só a forma de calcular: o streaming vai somando aos poucos conforme cada mensagem chega, enquanto o Pandas carrega tudo de uma vez e calcula a média em lote. O caminho é diferente, mas o resultado final tem que ser o mesmo (e foi).

**Desafio opcional — e se tivesse duas instâncias do consumidor no mesmo grupo?**

Nesse caso o Kafka ia dividir as 3 partições entre as duas instâncias, por exemplo uma ficando com 2 partições e a outra com 1. O problema é que cada instância guarda a `soma` e a `contagem` só na memória dela mesma — então nenhuma das duas ia enxergar o total geral, só a fatia de mensagens que caiu nas partições que ela recebeu. Isso quer dizer que a média calculada por cada instância separadamente **não** ia bater com a média real de cada sensor, porque as leituras de um mesmo sensor podem cair em partições diferentes dependendo de como a chave foi distribuída pelo producer.

Isso mostra um problema bem clássico de agregação com estado em sistemas distribuídos: o estado fica picado entre as partições/instâncias e não existe uma visão global automática. É basicamente o motivo de ferramentas como Kafka Streams e Spark Streaming existirem — elas cuidam de manter um estado por partição e, quando precisa, fazem um reshuffle das chaves pra garantir que tudo que é do mesmo sensor caia na mesma tarefa, ou usam algum banco/cache compartilhado pra juntar os resultados parciais depois. Sem esse tipo de mecanismo, uma agregação feita "na mão" com consumidores paralelos só fica correta dentro de cada partição isolada, não no todo.
